In [1]:
import os, sys
from tqdm import tqdm
import torch
import numpy as np

sys.path.insert(0, os.path.join(os.path.abspath(''), '..'))
from cmm.ffxml import ForceFieldXML
from cmm.topology import Topology
from cmm.units import BOHR2NM, BOHR2ANG

import openmm.app as app
from ase.io import read

In [2]:
torch.set_default_dtype(torch.float64)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
ff_path = os.path.join(os.path.abspath(''), '../scripts/water_refit.xml')
ff = ForceFieldXML(ff_path, device=device)



In [3]:
water_cluster_path = os.path.join(os.path.abspath(''), '../tests/data/water_clusters/all_reference_clusters.pdb')

In [14]:
water_cluster_pdb = app.PDBFile(water_cluster_path)
positions = [water_cluster_pdb.getPositions(True, frame=i)._value * 10.0 for i in range(water_cluster_pdb.getNumFrames())]

topologies = [Topology.fromMultiPDB(water_cluster_path, device, frame_index=i) for i in range(water_cluster_pdb.getNumFrames())]
systems = [ff.parametrize(topology, use_lr_dispersion=False, use_cutoff=False) for topology in topologies]

In [15]:
coords = torch.from_numpy(positions[0] / BOHR2ANG).to(device).requires_grad_(False)
natoms = coords.shape[0]
box = torch.tensor(np.eye(3) * 100.0, requires_grad=False, device=device)
systems[0].getEnergy(coords, box)

NotImplementedError: No ewald is not supported